# Extract column inputs for 1d model

Extract 1-D column-model inputs at 0°, 140°W from ACCESS-OM2 runs.

Writes `euc_column_inputs_140W.nc` with, for each run in `RUNS` (`ctrl` = era5, `warm` = tas_053, `wind` = uas_015):

- `T(z), S(z), u(z), w(z)` : 2S–2N band means at 140W [°C, psu, m/s, m/s], all on the T levels
- `pgf(z)` : −(1/ρ₀) dp/dx from pot_rho_0 + sea_level, slope over `PGF_LON` [m/s²]
- `tau_x, q_net, sst` : scalars at 140W band mean [N/m², W/m², °C]

averaged over `YEARS[run]` (last decade).

The wind-only run is used by `euc_column_kpp.jl` to split the warming run's PGF change into a
stress-attributable part (wind-run ΔPGF × Δτ_warm/Δτ_wind) and a pycnocline part.

`load_vars(base)` returns a dict with: `u, wt, pottemp, prac_salt, potrho, sea_level, tau_x, net_sfc_heating`.
Longitudes in ACCESS native (−280..80). `wt` is relabelled from `sw_ocean` to `st_ocean` but keeps w-level
depths; `column_inputs` prepends w = 0 at the surface and interpolates onto the T levels.

In [ ]:
import numpy as np, xarray as xr

LON_EVAL = 220.0                      # 140W (0-360)
PGF_LON  = (210.0, 230.0)             # 150W-130W
LAT      = slice(-2, 2)
ZMAX     = 500.0
RHO0, G  = 1026.0, 9.81
RUNS     = {"ctrl": "era5", "warm": "tas_053", "wind": "uas_015"}   # wind-only run calibrates the stress response
YEARS    = {k: slice("2008", "2017") for k in RUNS}                 # last decade of each

nat = lambda x: x - 360 if x > 80 else x      # 0-360 -> ACCESS native (-280..80)

def _band(da, latname, lonname, lon=None, lonslice=None):
    da = da.sel({latname: LAT}).mean(latname)
    if lon is not None:
        da = da.sel({lonname: nat(lon)}, method="nearest")
    if lonslice is not None:
        da = da.sel({lonname: slice(nat(lonslice[0]), nat(lonslice[1]))})
    return da

In [ ]:
import glob
import xarray as xr

def load_vars(base):
    ds = {}

    def mf(pattern):
        return xr.open_mfdataset(glob.glob(base + pattern), decode_timedelta=False)

    ds["sea_level"]       = mf('output*/ocean/ocean-2d-sea_level-1-monthly-mean-*.nc').sea_level
    ds["tau_x"]           = mf('output*/ocean/ocean-2d-tau_x-1-monthly-mean-*.nc').tau_x
    ds["net_sfc_heating"] = mf('output*/ocean/ocean-2d-net_sfc_heating-1-monthly-mean-*.nc').net_sfc_heating

    ds["u"]  = mf('output*/ocean/ocean-3d-u-1-monthly-mean-*.nc').u
    ds["v"]  = mf('output*/ocean/ocean-3d-v-1-monthly-mean-*.nc').v

    # w-grid relabelled onto tracer-centre names; values sit half a cell deep.
    ds["wt"] = mf('output*/ocean/ocean-3d-wt-1-monthly-mean-*.nc').wt.rename({'sw_ocean': 'st_ocean'})

    ds["potrho"]    = mf('output*/ocean/ocean-3d-pot_rho_0-1-monthly-mean-*.nc').pot_rho_0
    ds["pottemp"]   = mf('output*/ocean/ocean-3d-pot_temp-1-monthly-mean-*.nc').pot_temp
    ds["prac_salt"] = mf('output*/ocean/ocean-3d-salt-1-monthly-mean-*.nc').salt      # needed for S and g'

    return ds


DATASETS = {
    "era5":    "/g/data/e14/ic0706/access-om2/1deg_era5_iaf/",
    "tas_053": "/scratch/e14/ic0706/access-om2/archive/1deg_era5_iaf_tas_053_increase/",
    "uas_015": "/scratch/e14/ic0706/access-om2/archive/1deg_era5_iaf_uas_015_increase/",   # <- check this path
}

In [ ]:
import os
for name, base in DATASETS.items():
    print(name, base, "exists:", os.path.isdir(base))
    print("  outputs:", sorted(glob.glob(base + "output*"))[:3], "...")
    print("  sea_level files:", len(glob.glob(base + "output*/ocean/ocean-2d-sea_level-1-monthly-mean-*.nc")))

In [ ]:
def _get(ds, *names):
    for n in names:
        if n in ds:
            return ds[n]
    raise KeyError(f"none of {names} in load_vars output")

def column_inputs(ds, years):
    sel = lambda da: da.sel(time=years)
    u_, w_ = sel(_get(ds, "u")), sel(_get(ds, "wt"))
    pt     = sel(_get(ds, "pottemp", "pot_temp"))
    sp     = sel(_get(ds, "prac_salt", "salt"))
    rho_   = sel(_get(ds, "potrho", "pot_rho_0"))
    eta_   = sel(_get(ds, "sea_level"))
    taux_  = sel(_get(ds, "tau_x"))
    qnet_  = sel(_get(ds, "net_sfc_heating"))
    if float(pt.isel(time=0).max()) > 100: pt = pt - 273.15

    # --- profiles at 140W, 2S-2N, time mean; everything ends up on the T levels -------------
    T  = _band(pt, "yt_ocean", "xt_ocean", lon=LON_EVAL).sel(st_ocean=slice(0, ZMAX)).mean("time").load()
    S  = _band(sp, "yt_ocean", "xt_ocean", lon=LON_EVAL).sel(st_ocean=slice(0, ZMAX)).mean("time").load()
    zt = T["st_ocean"].values

    u  = _band(u_, "yu_ocean", "xu_ocean", lon=LON_EVAL).sel(st_ocean=slice(0, ZMAX)).mean("time").load()
    u  = u.interp(st_ocean=zt)                        # same levels as T; harmless

    # w is on w-levels (sw_ocean, or sw_ocean relabelled as st_ocean with DIFFERENT values).
    # Prepend w = 0 at the surface, then interpolate onto the T levels.
    w  = _band(w_, "yt_ocean", "xt_ocean", lon=LON_EVAL).mean("time").load()
    if "sw_ocean" in w.dims:
        w = w.rename({"sw_ocean": "st_ocean"})
    w0 = xr.zeros_like(w.isel(st_ocean=0)).assign_coords(st_ocean=0.0)
    w  = xr.concat([w0, w], "st_ocean").interp(st_ocean=zt)

    # --- pressure gradient force: p/rho0 = g*eta + (g/rho0) * int_z^0 (rho - rho0) dz' --------
    rho = _band(rho_, "yt_ocean", "xt_ocean", lonslice=PGF_LON).sel(st_ocean=slice(0, ZMAX)).mean("time").load()
    eta = _band(eta_, "yt_ocean", "xt_ocean", lonslice=PGF_LON).mean("time").load()
    if float(rho.max()) < 500: rho = rho + 1000.0                                  # sigma -> rho
    dz  = xr.ones_like(rho["st_ocean"]) * np.gradient(rho["st_ocean"].values)      # layer thickness (approx)
    phyd = (G / RHO0) * ((rho - RHO0) * dz).cumsum("st_ocean") - (G / RHO0) * 0.5 * (rho - RHO0) * dz
    p_over_rho = G * eta + phyd                                                    # (st_ocean, xt_ocean)
    xm = np.deg2rad(p_over_rho["xt_ocean"].values) * 6.371e6                        # metres along equator
    slope = np.polyfit(xm, p_over_rho.transpose("st_ocean", "xt_ocean").values.T, 1)[0]   # d(p/rho)/dx per level
    pgf = xr.DataArray(-slope, coords={"st_ocean": p_over_rho["st_ocean"]}, dims="st_ocean")
    pgf = pgf.interp(st_ocean=zt)

    tau  = float(_band(taux_, "yu_ocean", "xu_ocean", lon=LON_EVAL).mean("time"))
    qnet = float(_band(qnet_, "yt_ocean", "xt_ocean", lon=LON_EVAL).mean("time"))
    sst  = float(T.isel(st_ocean=0))

    out = xr.Dataset(dict(T=T, S=S, u=u, w=w, pgf=pgf)).reset_coords(drop=True)
    out = out.assign(tau_x=tau, q_net=qnet, sst=sst)
    bad = out.to_array().isnull()
    assert int(bad.sum()) == 0, f"NaNs in column inputs at st_ocean={out['st_ocean'].values[bad.any('variable').values]}"
    return out

In [ ]:
OUT = "euc_column_inputs_140W.nc"   # copy to the repo's inputs/ when done

pieces = []
for key, name in RUNS.items():
    print(f"=== {key} ({name}) ===", flush=True)
    ds  = load_vars(DATASETS[name])
    out = column_inputs(ds, YEARS[key]).expand_dims(run=[key])
    pieces.append(out)
    print(key, "done:",
          f"{out.sizes['st_ocean']} levels,",
          f"core ~{float(out.u.idxmax('st_ocean').squeeze()):.0f} m,",
          f"tau {float(out.tau_x.squeeze()):+.3f} N/m2,",
          f"SST {float(out.sst.squeeze()):.2f} C", flush=True)

inputs = xr.concat(pieces, "run")
assert set(inputs["run"].values) == set(RUNS), f"runs written: {inputs['run'].values}"
assert int(inputs.to_array().isnull().sum()) == 0, "NaNs in output"
inputs.to_netcdf(OUT)
print("wrote", OUT, "with runs", list(inputs["run"].values))

# quick sanity on the stress split the Julia script will use
tc, tw, tx = (float(inputs.tau_x.sel(run=r)) for r in ("ctrl", "warm", "wind"))
print(f"tau ctrl {tc:+.4f}  warm {tw:+.4f} ({(tw-tc)/abs(tc)*100:+.1f}%)  wind {tx:+.4f} ({(tx-tc)/abs(tc)*100:+.1f}%)"
      f"  -> dtau_warm/dtau_wind = {(tw-tc)/(tx-tc):.2f}")
inputs